# DMOSpeech 2 — EXP-028: Transmissible Slow-Band Bridge (`TBRG`) & `STAC-TBRG` Handover

- **Task ID:** `EVAL-021` (follows `EXP-027` / `EVAL-020`)
- **Estimated T4 Runtime:** ~25–30 minutes (reuses locked `EXP-027` fresh-speaker outputs for `student4`, `W4`, `W4R2`, `W8`, `W8R2`, `T16`).
- **Method Tested (`TBRG`, 13 NFE = -38.1% vs `W8`, -35.0% vs `W8R2`, -59.4% vs `T16`):**
  1. Run Teacher Steps `0..4` (`10 NFE`) -> `T4`, drifts `d_1..d_4`, and STAC cumulative action `A_4 = sum d_j`.
  2. Run Student `tau_1 = 0.0761` pass (`1 NFE`) -> clean acoustic draft `S1`.
  3. Inject Frobenius-matched slow prosody of `T4` into `S1`'s `voice + fine` bands before `tau_2 = 0.2929` (`0 NFE`), plus unnormalized ablation (`STS2`, 13 NFE), then finish `[tau_2, tau_3]` (`2 NFE`).
  4. Evaluate 5-fold speaker-CV adaptive routing `STAC-TBRG` (`TBRG` 13 NFE vs `W8R2` 20 NFE using Step-4 velocity `r_4 = d_4 / A_4`).


In [ ]:
import os
import subprocess
import torch

subprocess.run(["nvidia-smi"], check=True)
assert torch.cuda.is_available(), "CUDA GPU required; switch Colab runtime to T4 GPU."
print("GPU:", torch.cuda.get_device_name(0))

from google.colab import drive
drive.mount("/content/drive")

THESIS_ROOT = "/content/Final-Thesis-Setup"
DMO_ROOT = "/content/Shonget-DMOSpeech2"
OUT_DIR = f"{THESIS_ROOT}/research/experiments/EXP-028-transmissibility-bridge/results"
DRIVE_DIR = "/content/drive/MyDrive/exp028_transmissibility_bridge"
DRIVE_CACHE = "/content/drive/MyDrive/dmo_cache"
os.makedirs(DRIVE_DIR, exist_ok=True)
os.makedirs(DRIVE_CACHE, exist_ok=True)
print("Drive directories ready:", DRIVE_DIR)

## 1. Clone Repositories, Install Dependencies & Verify Preregistration

Clones `Shonget-DMOSpeech2` (pinned commit `8a7868fc...`) and `Final-Thesis-Setup` (branch `rhnl-3`). If private, add `GITHUB_TOKEN` to Colab Secrets.


In [ ]:
import json
import os
import subprocess
import sys

THESIS_BRANCH = "rhnl-3"
DMO_COMMIT = "8a7868fc3345702a586173447f4869205b501f26"
DMO_REPO = "https://github.com/Shonget-Thesis/Shonget-DMOSpeech2.git"
THESIS_ROOT = "/content/Final-Thesis-Setup"
DMO_ROOT = "/content/Shonget-DMOSpeech2"
STAGE = "both"  # 'both', 'diversity', or 'quality'

token = ""
try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN") or ""
except Exception:
    pass
thesis_url = f"https://{token + '@' if token else ''}github.com/Shonget-Thesis/Final-Thesis-Setup.git"

if not os.path.exists(DMO_ROOT):
    subprocess.run(["git", "clone", DMO_REPO, DMO_ROOT], check=True)
subprocess.run(["git", "-C", DMO_ROOT, "fetch", "-q", "origin"], check=True)
subprocess.run(["git", "-C", DMO_ROOT, "checkout", "-q", DMO_COMMIT], check=True)
subprocess.run(["git", "-C", DMO_ROOT, "log", "--oneline", "-1"], check=True)

if not os.path.exists(THESIS_ROOT):
    subprocess.run(["git", "clone", "-b", THESIS_BRANCH, thesis_url, THESIS_ROOT], check=True)
subprocess.run(["git", "-C", THESIS_ROOT, "fetch", "-q", "origin", THESIS_BRANCH], check=True)
subprocess.run(["git", "-C", THESIS_ROOT, "checkout", "-q", THESIS_BRANCH], check=True)
subprocess.run(["git", "-C", THESIS_ROOT, "pull", "-q", "origin", THESIS_BRANCH], check=True)
subprocess.run(["git", "-C", THESIS_ROOT, "log", "--oneline", "-1"], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "numpy==2.2.3", "pandas==2.2.3"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{DMO_ROOT}/requirements.txt"], check=True)
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "torchdiffeq", "vocos", "speechbrain", "transformers", "jiwer",
    "soundfile", "librosa", "pyarrow", "scipy", "jieba", "accelerate", "huggingface_hub"
], check=True)

prereg_path = f"{THESIS_ROOT}/research/experiments/EXP-028-transmissibility-bridge/preregistration.json"
assert os.path.exists(prereg_path), f"Missing {prereg_path} on branch {THESIS_BRANCH}"
prereg = json.loads(open(prereg_path, encoding="utf-8").read())
print(f"Preregistration verified: sha256={prereg['sha256'][:12]}..., div={prereg['n_fresh_div']}, qual={prereg['n_fresh_qual']}")

## 2. Checkpoints & Seed-TTS English Audio (Drive Cache Supported)

Restores checkpoints and dataset from `/content/drive/MyDrive/dmo_cache` if available, otherwise downloads from HuggingFace and verifies checkpoint SHA-256 digests.


In [ ]:
import hashlib
import os
import shutil
import subprocess

DMO_ROOT = "/content/Shonget-DMOSpeech2"
DRIVE_CACHE = "/content/drive/MyDrive/dmo_cache"
os.makedirs(f"{DMO_ROOT}/ckpts", exist_ok=True)
os.makedirs(f"{DRIVE_CACHE}/ckpts", exist_ok=True)

CKPTS = {
    "model_85000.pt": (
        "https://huggingface.co/yl4579/DMOSpeech2/resolve/main/model_85000.pt",
        "e143188f7c46b69abb60786937d79b89dac9777d3c4c1e91a97634bcd39ebf2c",
    ),
    "model_1500.pt": (
        "https://huggingface.co/yl4579/DMOSpeech2/resolve/main/model_1500.pt",
        "6bd50428f3b5f0f25c1a60a0053f40ee47c817bda03c8ee120100f22680ae4c2",
    ),
}

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 24), b""):
            h.update(block)
    return h.hexdigest()

for name, (url, expected) in CKPTS.items():
    local_p = f"{DMO_ROOT}/ckpts/{name}"
    cache_p = f"{DRIVE_CACHE}/ckpts/{name}"
    if not os.path.exists(local_p) and os.path.exists(cache_p):
        print(f"Restoring {name} from Drive cache...")
        shutil.copy2(cache_p, local_p)
    if not os.path.exists(local_p):
        subprocess.run(["wget", "-c", url, "-O", local_p], check=True)
    digest = sha256_file(local_p)
    assert digest == expected, f"{name} SHA-256 mismatch: got {digest}"
    if not os.path.exists(cache_p):
        shutil.copy2(local_p, cache_p)
    print(f"{name}: SHA-256 verified")

data_dir = f"{DMO_ROOT}/data/seed-tts-eval"
probe_wav = f"{data_dir}/en/prompt-wavs/common_voice_en_10933823.wav"
cache_data = f"{DRIVE_CACHE}/seed-tts-eval"
if (not os.path.exists(probe_wav) or os.path.getsize(probe_wav) < 1000) and os.path.exists(f"{cache_data}/en/meta.lst"):
    print("Restoring seed-tts-eval from Drive cache...")
    shutil.copytree(cache_data, data_dir, dirs_exist_ok=True)
if not os.path.exists(probe_wav) or os.path.getsize(probe_wav) < 1000:
    from huggingface_hub import snapshot_download
    shutil.rmtree(data_dir, ignore_errors=True)
    snapshot_download(repo_id="zhaochenyang20/seed-tts-eval", repo_type="dataset", local_dir=data_dir)
assert os.path.exists(probe_wav) and os.path.getsize(probe_wav) >= 1000, "Seed-TTS audio missing"
print("Seed-TTS audio verified [OK]")

## 3. Restore Any Partial Results from Google Drive

Restores prior `diversity.csv`, `quality.csv`, and `smoke/` from Drive if resuming.


In [ ]:
import csv
import os
import shutil

THESIS_ROOT = "/content/Final-Thesis-Setup"
OUT_DIR = f"{THESIS_ROOT}/research/experiments/EXP-028-transmissibility-bridge/results"
DRIVE_DIR = "/content/drive/MyDrive/exp028_transmissibility_bridge"
os.makedirs(OUT_DIR, exist_ok=True)

for rel in ("diversity.csv", "quality.csv", "smoke/diversity.csv", "smoke/quality.csv", "smoke/smoke_check.json"):
    src = os.path.join(DRIVE_DIR, rel)
    dst = os.path.join(OUT_DIR, rel)
    if os.path.exists(src) and (not os.path.exists(dst) or os.path.getsize(src) > os.path.getsize(dst)):
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        shutil.copy2(src, dst)
        print("Restored from Drive:", rel)

for fname, target in (("diversity.csv", 20), ("quality.csv", 40)):
    p = os.path.join(OUT_DIR, fname)
    n = sum(1 for _ in csv.DictReader(open(p, encoding="utf-8"))) if os.path.exists(p) else 0
    print(f"{fname}: {n} / {target} rows")

## 4. Pre-Flight Smoke Test (2 Fresh Speakers, Split-Student Identity & EXP-027 W4 Check)

Verifies preregistration & `EXP-027` CSV hashes, confirms `max |W4_split - W4| <= 1e-5` and `EXP-027` `W4` WER/SIM within `0.01`, and exercises `TBRG`/`STS2` diversity and quality loops.


In [ ]:
import json
import os
import subprocess
import sys

THESIS_ROOT = "/content/Final-Thesis-Setup"
DMO_ROOT = "/content/Shonget-DMOSpeech2"
OUT_DIR = f"{THESIS_ROOT}/research/experiments/EXP-028-transmissibility-bridge/results"
RUNNER = f"{THESIS_ROOT}/code/colab/run_transmissibility_bridge.py"

os.chdir(THESIS_ROOT)
subprocess.run([sys.executable, RUNNER, "smoke", "--dmo_root", DMO_ROOT, "--out_dir", OUT_DIR], check=True)
print(json.dumps(json.loads(open(f"{OUT_DIR}/smoke/smoke_check.json", encoding="utf-8").read()), indent=2))

## 5. Run Synthesis (`diversity`: 20 x 20 seeds; `quality`: 40 x 2 new arms)

Runs with a 60-second background Drive sync loop (~25–30 min total on T4).


In [ ]:
import os
import shutil
import subprocess
import sys
import threading

THESIS_ROOT = "/content/Final-Thesis-Setup"
DMO_ROOT = "/content/Shonget-DMOSpeech2"
OUT_DIR = f"{THESIS_ROOT}/research/experiments/EXP-028-transmissibility-bridge/results"
DRIVE_DIR = "/content/drive/MyDrive/exp028_transmissibility_bridge"
RUNNER = f"{THESIS_ROOT}/code/colab/run_transmissibility_bridge.py"
STAGE = globals().get("STAGE", "both")

def sync_to_drive():
    if not os.path.exists(OUT_DIR):
        return
    for root, _, files in os.walk(OUT_DIR):
        for fn in files:
            src = os.path.join(root, fn)
            rel = os.path.relpath(src, OUT_DIR)
            dst = os.path.join(DRIVE_DIR, rel)
            os.makedirs(os.path.dirname(dst), exist_ok=True)
            shutil.copy2(src, dst)

stop_evt = threading.Event()
def sync_loop():
    while not stop_evt.wait(60):
        try:
            sync_to_drive()
        except Exception as exc:
            print("Drive sync warning:", exc)

t = threading.Thread(target=sync_loop, daemon=True)
t.start()
os.chdir(THESIS_ROOT)
try:
    subprocess.run([
        sys.executable, RUNNER, "run",
        "--dmo_root", DMO_ROOT, "--out_dir", OUT_DIR, "--stage", STAGE,
    ], check=True)
finally:
    stop_evt.set()
    t.join(timeout=5)
    sync_to_drive()
print("Run finished and synced to Drive.")

## 6. Progress Check & Optional Partial Preview


In [ ]:
import csv
import os
import subprocess
import sys

THESIS_ROOT = "/content/Final-Thesis-Setup"
OUT_DIR = f"{THESIS_ROOT}/research/experiments/EXP-028-transmissibility-bridge/results"
RUNNER = f"{THESIS_ROOT}/code/colab/run_transmissibility_bridge.py"

counts = {}
for fname, target in (("diversity.csv", 20), ("quality.csv", 40)):
    p = os.path.join(OUT_DIR, fname)
    counts[fname] = sum(1 for _ in csv.DictReader(open(p, encoding="utf-8"))) if os.path.exists(p) else 0
    print(f"{fname}: {counts[fname]} / {target}")

if counts["diversity.csv"] >= 2 and counts["quality.csv"] >= 2 and (counts["diversity.csv"] < 20 or counts["quality.csv"] < 40):
    os.chdir(THESIS_ROOT)
    subprocess.run([sys.executable, RUNNER, "analyze", "--out_dir", OUT_DIR, "--allow_partial"], check=True)
    print(open(f"{OUT_DIR}/TRANSMISSIBILITY_BRIDGE_REPORT.md", encoding="utf-8").read())

## 7. Final Analysis, Report & Archive Export

Validates full 20/20 diversity & 40/40 quality completion, merges EXP-027 references, computes CIs and 5-fold `STAC-TBRG` routing, writes `MANIFEST.json`, syncs to Drive, and downloads the results ZIP.


In [ ]:
import os
import subprocess
import sys

THESIS_ROOT = "/content/Final-Thesis-Setup"
OUT_DIR = f"{THESIS_ROOT}/research/experiments/EXP-028-transmissibility-bridge/results"
RUNNER = f"{THESIS_ROOT}/code/colab/run_transmissibility_bridge.py"

os.chdir(THESIS_ROOT)
subprocess.run([sys.executable, RUNNER, "analyze", "--out_dir", OUT_DIR], check=True)
print(open(f"{OUT_DIR}/TRANSMISSIBILITY_BRIDGE_REPORT.md", encoding="utf-8").read())
print(open(f"{OUT_DIR}/transmissibility_bridge_results.json", encoding="utf-8").read()[:1200])

In [ ]:
import os
import shutil
import zipfile
from google.colab import files

THESIS_ROOT = "/content/Final-Thesis-Setup"
OUT_DIR = f"{THESIS_ROOT}/research/experiments/EXP-028-transmissibility-bridge/results"
DRIVE_DIR = "/content/drive/MyDrive/exp028_transmissibility_bridge"
ZIP_NAME = "DMOSpeech2_EXP028_Transmissibility_Bridge_Results.zip"
ZIP_PATH = f"/content/{ZIP_NAME}"

for root, _, fns in os.walk(OUT_DIR):
    for fn in fns:
        src = os.path.join(root, fn)
        rel = os.path.relpath(src, OUT_DIR)
        dst = os.path.join(DRIVE_DIR, rel)
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        shutil.copy2(src, dst)

with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zf:
    for root, _, fns in os.walk(OUT_DIR):
        for fn in fns:
            src = os.path.join(root, fn)
            zf.write(src, arcname=os.path.relpath(src, OUT_DIR))

shutil.copy2(ZIP_PATH, os.path.join(DRIVE_DIR, ZIP_NAME))
print("Packaged:", ZIP_PATH, f"({os.path.getsize(ZIP_PATH) / 1024:.1f} KB)")
files.download(ZIP_PATH)